# Laya V3.9 — Full Attention + TWO Sliding Blocks → ONE Global–CeNN Superlayer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V39_Global_CeNN_TwoSliding_3to1_Colab.ipynb)

### Design based on the TinyCeNN-LM experimental evidence

V3.8's zero-shot per-layer deletions mostly did not meet simultaneous accuracy and speed requirements; masks also seldom sped up already-fused native SDPA. Your prior **Jev_CeNN_2Window_3Layers_To_1_Colab** established that CeNN had a *causal* effect on held-out local reconstruction (cosine +0.0214 when enabled), but a 20.7M-parameter two-attention-window superlayer for layers 15–17 remained slow (18.36ms versus 5.78ms for three originals) and lost downstream accuracy (74.2% agreement). This notebook **does not** repeat that two-SDPA-window design.

**New proposal: preserve the expensive-to-approximate pretrained global attention once, then replace two successive pretrained sliding-attention + FFN blocks with a small shared-weight, two-step CeNN operator.** There is **no query-routed remote-key search, no per-query top-k SDPA, and no extra full-attention call**.

| Original | Replacement |
|---|---|
| Layer 0 global SDPA + FFN → 1 sliding SDPA + FFN → 2 sliding SDPA + FFN | ONE block at slot 0: exact frozen original global block → shared local CeNN step 1 → shared local CeNN step 2; slots 1 and 2 are parameter-free bypasses |
| Same triplets 3–5, 6–8, …, 24–26 | Same independently distilled CeNN *architecture*, shared across the two local steps **within each group** |
| Layer 27 (unpaired full attention) | Leave original unchanged |

Thus **nine original triples become nine active hybrid superlayers** plus the original final global block: 28 original full encoder blocks → **10 active encoder blocks** (in the existing 28-slot container, 18 slots bypassed). The nine global branches retain pretrained ModernBERT RoPE, QKV, SDPA and original MLP; the two local blocks' attention, norms and two MLPs are *learned* by CeNN adapters. The final layer stays untouched. This is a structural approximation, not mathematical equivalence.

### CeNN hypothesis and attention windows

A two-step CeNN cell uses the **same spatial weights and channel bottleneck** for both sequential steps, but separate trainable step gains. Each step contains three depthwise 1D cellular neighborhoods in a low-rank channel space:

- Fine window: kernel 7, dilation 1 (radius 3)
- Intermediate window: kernel 7, dilation 4 (radius 12)
- Wide window: kernel 9, dilation 16 (radius 64, matching ModernBERT's original sliding radius)

After channel mixing, the residual state is updated by the shared CeNN cell twice. The two windows here mean **two temporal CeNN steps**, not two full attention mechanisms. Two steps permit paths up to an effective ±128 neighborhood, subject to boundaries. They cannot generally reproduce an arbitrary pair of pretrained Q/K attention matrices, and learned weight similarity across consecutive sliding layers must be **measured**, not assumed.

### Training and proof

1. Build **one** pretrained Laya teacher on the T4; freeze its weights. Derive case-disjoint TRAIN, PROBE and AUDIT from source TRAIN only. Never use official TEST for selecting configurations.
2. For each of nine triples, capture **frozen teacher** post-global state, post-first-sliding state and post-second-sliding state; cache activations on CPU in FP16, train only that triple's small CeNN with intermediate and terminal *delta* supervision; checkpoint best PROBE state.
3. One-group control: directly disable CeNN's spatial path at inference, and optionally train an equal-step pointwise-only baseline. Inspect adjacent original sliding QKV/Wo weight cosine and compare segment functional outputs rather than claiming shared weights are identical.
4. Insert all nine trained Global–CeNN superlayers **simultaneously** (two bypasses each). Measure whole-model PROBE decisions and paired CUDA speed, then optionally perform short task-specific end-to-end distillation with original teacher probability vectors cached from TRAIN on CPU. This stage trains **only CeNN adapters**; the original global blocks, embeddings and classifier are frozen.
5. Evaluate AUDIT **once** after selecting the final checkpoint using PROBE only. Measure native versus hybrid group latency, end-to-end latency on identical batches, CeNN on/off, teacher agreement, gold accuracy and JS divergence. Save a reusable adapter-only checkpoint and a JSON report. **Never declare success from local cosine alone.**

### Defaults and limitations

- **Train nine groups** with 240 steps each and a small probe cache, optional end-to-end recovery (120 steps). This can take many minutes on a T4; Google Drive checkpointing is optional.
- **One pretrained checkpoint loaded**; CeNN adapters are small. Training snapshots are CPU-only and streamed to GPU.
- Acceptance requires ≥95% teacher decision agreement, ≤1 percentage point gold accuracy drop, ≥0.99 group-output cosine on audited segments, and repeatable >1× matched end-to-end speed. These are goals, not claimed results.
- The primary output is a **research checkpoint**, not a deployable model unless all gates pass. Do not assume CeNN must win; compare it with the original and with its spatial component disabled.

### Additional validation

For each original sliding pair, the experiment checks both **Wqkv/Wo parameter-vector cosine** and the cosine of their **teacher-measured residual updates**. This is intended to test the claim that the two sliding blocks learn similar interactions, rather than assuming it. It also adds an interleaved, matched-shape **A/B/B/A** full-model CUDA timing check and reports theoretical parameter savings separately from the still-unpruned research checkpoint.

### Optional speed-aware CeNN branch pruning

After each group is trained, V3.9 tests removing fine, intermediate, or wide depthwise convolution branches **without retraining** on PROBE only. It retains a smaller branch set only if the local teacher cosine drops by at most 0.002, residual normalized error rises by at most 0.015, and the reduced cell improves matched GPU latency by at least 5%. The selected active branch indices are saved with the adapter checkpoint, because `state_dict` alone does not encode runtime branch selection.


In [ ]:
#@title 1. Install pinned model dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2039
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 500 #@param {type:"integer"}
DEV_CASES = 240 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 2 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 1 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 2 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 2 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V39_GlobalCeNN_3to1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4 and TRAIN_CACHE_BATCHES > 0
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
# student_eval not built: use a single model and temporarily swap wrappers
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
# single teacher model only
teacher.encoder.config.reference_compile = False
# one model; reference_compile disabled above

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
#@title V3.9 trainable nine-group compression controls
CENN_RANK=96 #@param {type:"integer"}
GROUP_STEPS=240 #@param {type:"integer"}
GROUP_TRAIN_CACHE_BATCHES=32 #@param {type:"integer"}
GROUP_PROBE_CACHE_BATCHES=6 #@param {type:"integer"}
CHECK_EVERY=40 #@param {type:"integer"}
TRAIN_LEARNING_RATE=0.0012 #@param {type:"number"}
USE_SHARED_CELL=True #@param {type:"boolean"}
RUN_END2END_CALIBRATION=True #@param {type:"boolean"}
END2END_STEPS=120 #@param {type:"integer"}
TASK_TRAIN_BATCHES=40 #@param {type:"integer"}
TASK_LEARNING_RATE=0.00012 #@param {type:"number"}
RUN_POINTWISE_CONTROL=False #@param {type:"boolean"}
RUN_CENN_SPATIAL_ABLATION=True #@param {type:"boolean"}
RUN_SCALE_PRUNING=True #@param {type:"boolean"}
MAX_PROBE_COSINE_LOSS=.002 #@param {type:"number"}
MAX_PROBE_DELTA_NMSE_INCREASE=.015 #@param {type:"number"}
MIN_SCALE_SPEEDUP=1.05 #@param {type:"number"}
QUALITY_PROBE_DECISIONS=100 #@param {type:"integer"}
FINAL_AUDIT_DECISIONS=200 #@param {type:"integer"}
BENCH_REPEATS=8 #@param {type:"integer"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}
RESUME_GROUP_TRAINING=True #@param {type:"boolean"}
assert BATCH in (1,2,3) and GROUP_STEPS>=1 and CENN_RANK>=16
assert TRAIN_CASES+DEV_CASES<=1200 and CHECK_EVERY>=1
assert GROUP_TRAIN_CACHE_BATCHES>=1 and GROUP_PROBE_CACHE_BATCHES>=1
assert TASK_TRAIN_BATCHES>=1 and BENCH_REPEATS>=3
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR=Path("/content/drive/MyDrive/TinyCeNN-LM/V39_GlobalCeNN")
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
from tinycenn_lm.standalone_decision import full_attention_indices
FULL_LAYERS=full_attention_indices(teacher)
GROUPS=[(g,g+1,g+2) for g in FULL_LAYERS if g+2<len(teacher.encoder.layers)]
assert FULL_LAYERS==[0,3,6,9,12,15,18,21,24,27]
assert GROUPS==[(g,g+1,g+2) for g in range(0,27,3)]
assert teacher.encoder.config._attn_implementation=="sdpa"
assert int(teacher.encoder.config.local_attention)//2==64
print("Nine triples:",GROUPS,"unpaired full:",FULL_LAYERS[-1])
print("Exactly one pretrained model allocated. New CeNN rank",CENN_RANK)


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.9 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Small shared-weight two-step cellular block + strict ModernBERT contract
from contextlib import contextmanager
from functools import lru_cache

def mask_from_encoder(attention_mask,x):
    """Supports original ModernBERT [B,1,T,T] additive or [B,T] masks."""
    if attention_mask is None:
        return torch.ones(x.shape[:2],device=x.device,dtype=torch.bool)
    if attention_mask.ndim==2:
        return attention_mask.bool()
    if attention_mask.ndim!=4 or attention_mask.shape[-2:]!=(x.shape[1],x.shape[1]):
        raise ValueError(f"Bad attention mask {attention_mask.shape}")
    edge=attention_mask[:,0,0,:]
    return edge.bool() if edge.dtype==torch.bool else edge.float()>-1e4

class SharedCeNNCell(nn.Module):
    """Low-rank CeNN: pointwise channel interaction + 3 local spatial bands."""
    def __init__(self,hidden,rank=96,shared=True):
        super().__init__()
        self.hidden=hidden
        self.rank=rank
        self.shared=shared
        self.norm=nn.LayerNorm(hidden,eps=1e-5)
        self.down=nn.Linear(hidden,rank,bias=False)
        self.up=nn.Linear(rank,hidden,bias=False)
        self.pointwise=nn.Linear(rank,rank,bias=False)
        def bank():
            return nn.ModuleList([
                nn.Conv1d(rank,rank,7,padding=3,groups=rank,bias=False),
                nn.Conv1d(rank,rank,7,padding=12,dilation=4,groups=rank,bias=False),
                nn.Conv1d(rank,rank,9,padding=64,dilation=16,groups=rank,bias=False),
            ])
        self.spatial=bank()
        self.spatial2=None if shared else bank()
        self.alpha=nn.Parameter(torch.tensor([0.0,0.0],dtype=torch.float32))
        self.spatial_enabled=True
        self.enabled_scales=(0,1,2)
        self.last_mid=None
        nn.init.normal_(self.up.weight,std=0.002)
        for p in self.pointwise.parameters():
            nn.init.orthogonal_(p)

    def _step(self,state,valid,which):
        bmask=valid.unsqueeze(-1).to(state.dtype)
        low=self.down(self.norm(state)*bmask)
        spatial=torch.zeros_like(low)
        if self.spatial_enabled:
            layers=self.spatial if which==0 or self.shared else self.spatial2
            act=low.transpose(1,2).contiguous()
            for branch_index,conv in enumerate(layers):
                if branch_index in self.enabled_scales:
                    spatial=spatial+conv(act).transpose(1,2)
        new=F.silu(self.pointwise(low)+spatial+low)
        delta=self.up(new)
        gain=torch.sigmoid(self.alpha[which]).to(delta.dtype)
        return state+gain*delta*bmask

    def forward(self,x,valid,return_mid=False):
        valid=valid.bool()
        mid=self._step(x,valid,0)
        final=self._step(mid,valid,1)
        if return_mid:return mid,final
        return final

class BypassModernBertLayer(nn.Module):
    """Zero-parameter identity, exactly the layer tuple forward contract."""
    def forward(self,hidden_states,*args,**kwargs):
        return (hidden_states,)

class GlobalCeNNSuperLayer(nn.Module):
    """ONE active block = original frozen global block + 2 shared CeNN steps."""
    def __init__(self,original_global,adapter):
        super().__init__()
        self.global_block=original_global
        self.cenn=adapter
    def forward(self,hidden_states,attention_mask=None,*args,**kwargs):
        global_out=self.global_block(hidden_states,attention_mask=attention_mask,
                                     *args,**kwargs)[0]
        valid=mask_from_encoder(attention_mask,global_out)
        final=self.cenn(global_out,valid)
        return (final,)

ORIGINAL_ENCODER_LAYERS=tuple(teacher.encoder.layers)
STUDENT_CELLS=nn.ModuleDict()
for g,one,two in GROUPS:
    hidden=int(teacher.encoder.config.hidden_size)
    STUDENT_CELLS[str(g)]=SharedCeNNCell(
        hidden,CENN_RANK,shared=USE_SHARED_CELL
    ).to(device)

@contextmanager
def installed(groups):
    """Only insert trained student groups; global block original weights reused."""
    assert all(teacher.encoder.layers[i] is ORIGINAL_ENCODER_LAYERS[i]
               for i in range(len(ORIGINAL_ENCODER_LAYERS))),"Layers not restored!"
    try:
        for g in groups:
            assert g in [a for a,_,_ in GROUPS]
            teacher.encoder.layers[g]=GlobalCeNNSuperLayer(
                ORIGINAL_ENCODER_LAYERS[g],STUDENT_CELLS[str(g)])
            teacher.encoder.layers[g+1]=BypassModernBertLayer()
            teacher.encoder.layers[g+2]=BypassModernBertLayer()
        yield teacher
    finally:
        for i,module in enumerate(ORIGINAL_ENCODER_LAYERS):
            teacher.encoder.layers[i]=module
        assert all(teacher.encoder.layers[i] is ORIGINAL_ENCODER_LAYERS[i]
                   for i in range(len(ORIGINAL_ENCODER_LAYERS)))

assert len(ORIGINAL_ENCODER_LAYERS)==28 and FULL_LAYERS[-1]==27
assert sum(p.numel() for p in STUDENT_CELLS.parameters())<10_000_000
# Parameter accounting: only the trained adapters are new.
teacher_parameters=sum(p.numel() for p in teacher.parameters())
student_parameters=sum(p.numel() for p in STUDENT_CELLS.parameters())
removed_slide_parameters=sum(
    sum(p.numel() for p in ORIGINAL_ENCODER_LAYERS[i].parameters())
    for _,a,b in GROUPS for i in (a,b)
)
print("Original model preserved:",teacher_parameters/1e6,"M pretrained params")
print("Trainable nine CeNN adapters:",student_parameters/1e6,"M params")
print("Original sliding-layer params bypassed at runtime:",
      removed_slide_parameters/1e6,"M params")
print("Projected physically pruned parameter count (NOT yet exported):",
      (teacher_parameters-removed_slide_parameters+student_parameters)/1e6,"M")


In [ ]:
#@title 5. Frozen teacher capture for each triple, CPU-only caches and loss helpers
def cpu_half(t):
    if t is None:return None
    v=t.detach().cpu()
    return v.to(torch.float16).contiguous() if v.is_floating_point() else v.contiguous()

@torch.inference_mode()
def capture_teacher_triplet(g,items,n_batches,label="TRAIN"):
    cache=[]
    for n in range(n_batches):
        start=(n*BATCH)%len(items)
        selected=items[start:start+BATCH]
        if len(selected)<BATCH:selected+=items[:BATCH-len(selected)]
        b=direct_batch(selected)
        records={}
        def output(name):
            def post(mod,args,kwargs,out):
                records[name]=out[0].detach()
            return post
        hooks=[]
        for ix,key in ((g,"global"),(g+1,"first"),(g+2,"second")):
            hooks.append(teacher.encoder.layers[ix].register_forward_hook(
                output(key),with_kwargs=True
            ))
        try:
            with torch.autocast("cuda",dtype=amp_dtype):
                teacher(**b)
        finally:
            for hook in hooks:hook.remove()
        assert set(records)=={"global","first","second"}
        cache.append(dict(
            x=cpu_half(records["global"]),y1=cpu_half(records["first"]),
            y2=cpu_half(records["second"]),
            valid=b["attention_mask"].bool().cpu().contiguous()
        ))
    print(f"Teacher triple {g}-{g+2} {label} batches",len(cache),
          "stored on CPU in FP16")
    return cache

def to_gpu(entry):
    # Teacher caches were captured in torch.inference_mode(); those tensors
    # must be cloned *outside* inference mode before entering autograd.
    # Otherwise torch.nn.Linear can fail saving an inference tensor for
    # weight-gradient computation during group CeNN distillation.
    return {k:v.to(device,non_blocking=True).clone()
            for k,v in entry.items()}

def masked_nmse(pred,target,valid):
    mask=valid[:,:,None].float()
    e=((pred.float()-target.float()).square()*mask).sum()
    den=(target.float().square()*mask).sum().clamp_min(1e-5)
    return e/den

def cell_loss(pred1,pred2,e):
    mask=e["valid"]
    x=e["x"];y1=e["y1"];y2=e["y2"]
    # Normalizing residual learning avoids trivial high cosine from the skip
    # identity pathway, which can mask severe downstream classification drift.
    d1=masked_nmse(pred1-x,y1-x,mask)
    d2=masked_nmse(pred2-x,y2-x,mask)
    final=masked_nmse(pred2,y2,mask)
    intermediate=masked_nmse(pred1,y1,mask)
    return 0.45*d2+0.20*d1+0.25*final+0.10*intermediate

@torch.inference_mode()
def segment_quality(cell,cache,spatial_enabled=True):
    cell.eval()
    prior=cell.spatial_enabled
    cell.spatial_enabled=spatial_enabled
    err=total=delta_err=delta_power=cos_sum=tokens=0.
    try:
        with torch.autocast("cuda",dtype=amp_dtype):
            for row in cache:
                e=to_gpu(row)
                _,pred=cell(e["x"],e["valid"],return_mid=True)
                mask=e["valid"]
                m=mask[:,:,None].float()
                err+=float(((pred.float()-e["y2"].float()).square()*m).sum())
                total+=float((e["y2"].float().square()*m).sum())
                delta_err+=float((((pred-e["x"]).float()-
                                   (e["y2"]-e["x"]).float()).square()*m).sum())
                delta_power+=float((((e["y2"]-e["x"]).float()).square()*m).sum())
                cos_sum+=float(F.cosine_similarity(pred.float()[mask],
                                                   e["y2"].float()[mask],
                                                   dim=-1).sum())
                tokens+=int(mask.sum())
    finally:
        cell.spatial_enabled=prior
    return dict(cosine=cos_sum/max(tokens,1),
                nmse=err/max(total,1e-8),
                delta_nmse=delta_err/max(delta_power,1e-8),tokens=tokens)

def state_cpu(cell):
    return {k:v.detach().cpu().clone() for k,v in cell.state_dict().items()}

# Preflight: the exact global block can be reused inside a fused 3->1
# wrapper and the resulting attention tuple has the same dimensions.
preflight_batch=batch(probe_items,0,BATCH)
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    reference_logits,_=teacher(**preflight_batch)
with installed([GROUPS[0][0]]):
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        test_logits,_=teacher(**preflight_batch)
assert test_logits.shape==reference_logits.shape
assert torch.isfinite(test_logits).all()
assert teacher.encoder.layers[GROUPS[-1][-1]] is ORIGINAL_ENCODER_LAYERS[GROUPS[-1][-1]]
print("One hybrid 3->1 block installs, returns finite decision logits, and rolls back.")
print("Capture/train protocol ready for group 0-2 through 24-26.")


In [ ]:
#@title 6. Train all NINE 3→1 Global–CeNN groups, independently + resumably
group_history={}
group_probe={}
group_training_summary={}
for g,_,_ in GROUPS:
    cell=STUDENT_CELLS[str(g)]
    filename=OUTPUT_DIR/f"g{g:02d}_global_cenn_3to1.pt"
    if RESUME_GROUP_TRAINING and filename.exists():
        state=torch.load(filename,map_location="cpu",weights_only=True)
        assert state["rank"]==CENN_RANK and state["shared"]==USE_SHARED_CELL
        assert state["source_model"]==SOURCE_MODEL and state["group_start"]==g
        cell.load_state_dict(state["adapter"],strict=True)
        cell.eval().requires_grad_(False)
        group_training_summary[str(g)]=state["best_probe"]
        group_history[str(g)]=state["history"]
        print(f"G{g:02d} restored saved best-trained CeNN, probe",state["best_probe"])
        continue
    # Freeze other groups; train one small CeNN without touching pretrained backbone.
    train_cache=capture_teacher_triplet(
        g,train_items,GROUP_TRAIN_CACHE_BATCHES,"TRAIN")
    probe_cache=capture_teacher_triplet(
        g,probe_items,GROUP_PROBE_CACHE_BATCHES,"PROBE")
    cell.requires_grad_(True).train()
    optimizer=torch.optim.AdamW(cell.parameters(),lr=TRAIN_LEARNING_RATE,
                                weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=(amp_dtype==torch.float16))
    best=segment_quality(cell,probe_cache)
    best_rank=best["delta_nmse"]+.20*best["nmse"]
    best_state=state_cpu(cell)
    steps=[]
    attempt=0;updates=0
    while updates<GROUP_STEPS:
        if attempt>GROUP_STEPS+300:
            raise RuntimeError(f"G{g} repeated FP16 overflow; lower LR")
        e=to_gpu(train_cache[attempt%len(train_cache)])
        cell.train()
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=amp_dtype):
            mid,pred=cell(e["x"],e["valid"],return_mid=True)
            loss=cell_loss(mid,pred,e)
        require_finite(loss,f"V3.9 group {g} delta loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(cell.parameters(),1.0)
        oldscale=scaler.get_scale()
        scaler.step(optimizer);scaler.update()
        attempt+=1
        if scaler.is_enabled() and scaler.get_scale()<oldscale:continue
        updates+=1
        if updates==1 or updates%CHECK_EVERY==0 or updates==GROUP_STEPS:
            current=segment_quality(cell,probe_cache)
            score=current["delta_nmse"]+.20*current["nmse"]
            steps.append(dict(step=updates,**current))
            if score<best_rank:
                best_rank=score;best=current;best_state=state_cpu(cell)
            print(f"G{g:02d} step {updates:3}/{GROUP_STEPS}: "
                  f"probe cosine={current['cosine']:.5f} "
                  f"delta_nmse={current['delta_nmse']:.4f} "
                  f"best_cos={best['cosine']:.5f}")
    cell.load_state_dict(best_state,strict=True)
    cell.eval().requires_grad_(False)
    group_history[str(g)]=steps
    group_probe[str(g)]=best
    group_training_summary[str(g)]=best
    # Persist only lightweight trained adapter, no original teacher checkpoint.
    torch.save(dict(
        group_start=g,source_model=SOURCE_MODEL,rank=CENN_RANK,
        shared=USE_SHARED_CELL,best_probe=best,history=steps,
        adapter=best_state,
    ),filename)
    print("Saved",filename)
    del train_cache,probe_cache,optimizer,scaler,best_state
    torch.cuda.empty_cache()
print("All nine CeNN adapters available. No teacher weights changed.")


In [ ]:
#@title 7. Controlled CeNN spatial ablation and adjacent layer weight similarity
# Validate rather than ASSUME consecutive sliding windows learn identical weights.
group_weight_similarity={}
for g,a,b in GROUPS:
    local1=ORIGINAL_ENCODER_LAYERS[a]
    local2=ORIGINAL_ENCODER_LAYERS[b]
    similarities={}
    for key,m1,m2 in (
        ("Wqkv",local1.attn.Wqkv.weight,local2.attn.Wqkv.weight),
        ("Wo",local1.attn.Wo.weight,local2.attn.Wo.weight),
    ):
        similarities[key]=float(F.cosine_similarity(
            m1.detach().float().flatten()[None,:],
            m2.detach().float().flatten()[None,:],dim=1
        ).item())
    # Functional similarity is more meaningful than vector cosine: compare
    # the residual update from consecutive original sliding blocks on
    # the *same teacher trajectory*, with key padding removed.
    pair_cache=capture_teacher_triplet(g,probe_items,1,"INTERACTION_CHECK")
    sample=to_gpu(pair_cache[0])
    ok=sample["valid"]
    first_delta=(sample["y1"]-sample["x"]).float()[ok]
    second_delta=(sample["y2"]-sample["y1"]).float()[ok]
    similarities["delta_cosine"]=float(F.cosine_similarity(
        first_delta,second_delta,dim=-1).mean())
    similarities["delta_relative_energy"]=float(
        second_delta.square().mean()/first_delta.square().mean().clamp_min(1e-8)
    )
    group_weight_similarity[str(g)]=similarities
    print("Pair",a,b,"parameter and functional interaction:",similarities)
    del pair_cache,sample

cenn_ablation={}
if RUN_CENN_SPATIAL_ABLATION:
    for g,_,_ in GROUPS:
        probe_cache=capture_teacher_triplet(g,probe_items,2,"CENN_AUDIT_PROBE")
        cell=STUDENT_CELLS[str(g)]
        on=segment_quality(cell,probe_cache,True)
        off=segment_quality(cell,probe_cache,False)
        cenn_ablation[str(g)]=dict(
            spatial_on=on,spatial_off=off,
            spatial_cosine_gain=on["cosine"]-off["cosine"],
            delta_nmse_reduction=off["delta_nmse"]-on["delta_nmse"]
        )
        print(f"G{g:02d} spatial CeNN on vs off:",
              cenn_ablation[str(g)]["spatial_cosine_gain"])
        del probe_cache

# Training an equal-step non-spatial model isolates spatial weight contribution
# beyond the mere extra number of parameters.
pointwise_control=None
if RUN_POINTWISE_CONTROL:
    g=GROUPS[0][0]
    ctrl=SharedCeNNCell(
        int(teacher.encoder.config.hidden_size),CENN_RANK,
        shared=USE_SHARED_CELL
    ).to(device)
    ctrl.spatial_enabled=False
    train_cache=capture_teacher_triplet(g,train_items,
                                        GROUP_TRAIN_CACHE_BATCHES,"CONTROL_TRAIN")
    probe_cache=capture_teacher_triplet(g,probe_items,
                                        GROUP_PROBE_CACHE_BATCHES,"CONTROL_PROBE")
    opt=torch.optim.AdamW([p for n,p in ctrl.named_parameters()
                           if not n.startswith(("spatial.","spatial2."))],
                          lr=TRAIN_LEARNING_RATE)
    sc=torch.amp.GradScaler("cuda",enabled=(amp_dtype==torch.float16))
    best=None
    for step in range(GROUP_STEPS):
        e=to_gpu(train_cache[step%len(train_cache)])
        ctrl.train();opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=amp_dtype):
            m,y=ctrl(e["x"],e["valid"],return_mid=True)
            loss=cell_loss(m,y,e)
        sc.scale(loss).backward();sc.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(ctrl.parameters(),1.)
        sc.step(opt);sc.update()
    best=segment_quality(ctrl,probe_cache,False)
    pointwise_control=dict(group_start=g,steps=GROUP_STEPS,probe=best)
    print("Equal-step pointwise-only control:",pointwise_control)
    del ctrl,opt,sc,train_cache,probe_cache
    torch.cuda.empty_cache()

# A local three-scale CeNN can still perform redundant operations.
# Prune a scale ONLY if the trained group keeps close teacher fidelity
# AND measured GPU latency improves. All selection uses PROBE data only.
scale_pruning={}
if RUN_SCALE_PRUNING:
    def time_cell(cell,entry,repeats=6):
        with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
            for _ in range(3):cell(entry["x"],entry["valid"])
            torch.cuda.synchronize()
            ticks=[]
            for _ in range(repeats):
                a=torch.cuda.Event(enable_timing=True)
                b=torch.cuda.Event(enable_timing=True)
                a.record();cell(entry["x"],entry["valid"]);b.record()
                b.synchronize()
                ticks.append(float(a.elapsed_time(b)))
        return float(np.median(ticks))
    for g,_,_ in GROUPS:
        cell=STUDENT_CELLS[str(g)]
        probe_cache=capture_teacher_triplet(
            g,probe_items,2,"SPATIAL_SCALE_PROBE")
        entry=to_gpu(probe_cache[0])
        cell.spatial_enabled=True
        cell.enabled_scales=(0,1,2)
        original_quality=segment_quality(cell,probe_cache)
        original_ms=time_cell(cell,entry)
        candidate_rows=[]
        for active in ((0,2),(1,2),(0,1),(2,),(1,),(0,)):
            cell.enabled_scales=active
            metric=segment_quality(cell,probe_cache)
            acceptable=(
                metric["cosine"]>=original_quality["cosine"]-
                    MAX_PROBE_COSINE_LOSS and
                metric["delta_nmse"]<=original_quality["delta_nmse"]+
                    MAX_PROBE_DELTA_NMSE_INCREASE
            )
            if acceptable:
                ms=time_cell(cell,entry)
                candidate_rows.append(dict(
                    scales=list(active),quality=metric,ms=ms,
                    speedup=original_ms/max(ms,1e-9)))
        candidates=[row for row in candidate_rows
                    if row["speedup"]>=MIN_SCALE_SPEEDUP]
        if candidates:
            best=min(candidates,key=lambda x:x["ms"])
            choice=tuple(best["scales"])
            print("SCALE PRUNE G",g,"scales",choice,
                  "local speedup",round(best["speedup"],3),
                  "probe cosine",round(best["quality"]["cosine"],5))
        else:
            choice=(0,1,2)
            print("SCALE KEEP G",g,"all branches (insufficient speed/quality)")
        cell.enabled_scales=choice
        scale_pruning[str(g)]=dict(
            original_scales=[0,1,2],
            selected_scales=list(choice),
            original_quality=original_quality,
            original_ms=original_ms,
            candidates=candidate_rows)
        del probe_cache,entry
else:
    print("Scale pruning disabled; retain all fine/mid/wide CeNN branches.")


In [ ]:
#@title 8. Cache teacher decision distributions on disjoint TRAIN, PROBE and AUDIT cases
@torch.inference_mode()
def oracle_probs(items,limit):
    out=[]
    count=min(limit,len(items))
    for offset in range(0,count,BATCH):
        selection=items[offset:min(offset+BATCH,count)]
        data=direct_batch(selection)
        with torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**data)
        allowed=data["marker_mask"].bool()
        p=torch.softmax(logits.float().masked_fill(~allowed,-1e9),dim=-1)
        for i,row in enumerate(selection):
            out.append(dict(probs=p[i].detach().cpu(),
                            teacher=int(p[i].argmax()),
                            gold=int(row["gold_index"]),
                            case_id=row["case_id"],qid=row["qid"]))
    return out

# Split at original case level into probe and audit (already done in helper).
probe_truth=oracle_probs(probe_items,QUALITY_PROBE_DECISIONS)
audit_truth=oracle_probs(audit_items,FINAL_AUDIT_DECISIONS)
assert set(r["case_id"] for r in probe_truth).isdisjoint(
    set(r["case_id"] for r in audit_truth))
print("Native teacher PROBE",len(probe_truth),"AUDIT",len(audit_truth),
      "gold",sum(x["teacher"]==x["gold"] for x in audit_truth)/len(audit_truth))


In [ ]:
#@title 9. All nine triples installed simultaneously: whole-model PROBE evaluation
@torch.inference_mode()
def compare_decisions(items,reference):
    count=min(len(items),len(reference))
    matches=t_correct=s_correct=0
    total_js=0.
    differences=[]
    for start in range(0,count,BATCH):
        selected=items[start:min(start+BATCH,count)]
        data=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            scores,_=teacher(**data)
        mask=data["marker_mask"].bool()
        p=torch.softmax(scores.float().masked_fill(~mask,-1e9),dim=-1).cpu()
        for j,row in enumerate(selected):
            idx=start+j
            native=reference[idx]["probs"];new=p[j]
            assert len(native)==len(new)
            m=(native+new)/2
            total_js+=float(.5*((native*(native.clamp_min(1e-8).log()-
                                         m.clamp_min(1e-8).log())).sum()+
                                   (new*(new.clamp_min(1e-8).log()-
                                         m.clamp_min(1e-8).log())).sum()))
            guess=int(new.argmax()); gold=row["gold_index"]
            matches+=int(guess==reference[idx]["teacher"])
            t_correct+=int(reference[idx]["teacher"]==gold)
            s_correct+=int(guess==gold)
            if guess!=reference[idx]["teacher"]:
                differences.append(dict(case_id=row["case_id"],qid=row["qid"],
                                        teacher=int(reference[idx]["teacher"]),
                                        student=guess,gold=int(gold)))
    return dict(decisions=count,teacher_agreement=matches/count,
                teacher_gold_accuracy=t_correct/count,
                student_gold_accuracy=s_correct/count,
                gold_accuracy_drop=(t_correct-s_correct)/count,
                mean_js=total_js/count,changed_decisions=differences[:25])

GROUP_STARTS=[g for g,_,_ in GROUPS]
with installed(GROUP_STARTS):
    assert sum(isinstance(layer,GlobalCeNNSuperLayer)
               for layer in teacher.encoder.layers)==9
    assert sum(isinstance(layer,BypassModernBertLayer)
               for layer in teacher.encoder.layers)==18
    probe_pre_calibration=compare_decisions(probe_items,probe_truth)
print("Nine-group 28-slot compressed PROBE:",probe_pre_calibration)


In [ ]:
#@title 10. Optional end-to-end decision-level calibration of ONLY the nine CeNN cells
# Original global blocks, MLPs, embeddings and classifier stay frozen.
# Cache the teacher probabilities on TRAIN cases BEFORE installing students.
calibration_history=[]
task_cache=[]
if RUN_END2END_CALIBRATION:
    for i in range(TASK_TRAIN_BATCHES):
        selected=train_items[i*BATCH:(i+1)*BATCH]
        if len(selected)!=BATCH:break
        b=direct_batch(selected)
        with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**b)
        valid=b["marker_mask"].bool()
        t=torch.softmax(logits.float().masked_fill(~valid,-1e9),dim=-1)
        # Keep token inputs, marker mask, and teacher distribution on CPU.
        task_cache.append({
            "batch":{k:v.detach().cpu() for k,v in b.items()},
            "teacher_prob":t.detach().cpu(),
            "gold_labels":torch.tensor(
                [item["gold_index"] for item in selected],
                dtype=torch.long
            ),
        })
    assert len(task_cache)>=5,"Need enough TRAIN batches for end-to-end calibration"
    for c in STUDENT_CELLS.values():c.train().requires_grad_(True)
    opt=torch.optim.AdamW(STUDENT_CELLS.parameters(),
                          lr=TASK_LEARNING_RATE,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=(amp_dtype==torch.float16))
    best_probe=probe_pre_calibration
    best_score=best_probe["mean_js"]+2*(1-best_probe["teacher_agreement"])
    best_states={g:state_cpu(STUDENT_CELLS[g]) for g in STUDENT_CELLS}
    updates=0;attempts=0
    with installed(GROUP_STARTS):
        while updates<END2END_STEPS:
            if attempts>END2END_STEPS+300:
                raise RuntimeError("Too many unstable end-to-end CeNN updates")
            entry=task_cache[attempts%len(task_cache)]
            b={k:v.to(device) for k,v in entry["batch"].items()}
            target=entry["teacher_prob"].to(device)
            for c in STUDENT_CELLS.values():c.train()
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda",dtype=amp_dtype):
                student_logits,_=teacher(**b)
                choices=b["marker_mask"].bool()
                logp=F.log_softmax(student_logits.float().masked_fill(
                    ~choices,-1e9),dim=-1)
                # Teacher probability matching + small supervised gold term.
                kl=F.kl_div(logp,target,reduction="batchmean")
                labels=entry["gold_labels"].to(device)
                loss=kl+0.05*F.nll_loss(logp,labels)
            require_finite(loss,"V3.9 end-to-end task KL")
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(STUDENT_CELLS.parameters(),1.)
            prior=scaler.get_scale()
            scaler.step(opt);scaler.update()
            attempts+=1
            if scaler.is_enabled() and scaler.get_scale()<prior:continue
            updates+=1
            if updates==1 or updates%30==0 or updates==END2END_STEPS:
                for c in STUDENT_CELLS.values():c.eval()
                current=compare_decisions(probe_items,probe_truth)
                rank=current["mean_js"]+2*(1-current["teacher_agreement"])
                calibration_history.append(dict(step=updates,probe=current))
                if rank<best_score:
                    best_probe=current;best_score=rank
                    best_states={g:state_cpu(STUDENT_CELLS[g])
                                 for g in STUDENT_CELLS}
                print(f"Task calibration {updates}/{END2END_STEPS}: "
                      f"PROBE agreement={current['teacher_agreement']:.4f} "
                      f"JS={current['mean_js']:.5f} "
                      f"best={best_probe['teacher_agreement']:.4f}")
    for g,state in best_states.items():
        STUDENT_CELLS[g].load_state_dict(state,strict=True)
    del opt,scaler,best_states,task_cache
    torch.cuda.empty_cache()
else:
    print("End-to-end calibration disabled; using segment-distilled adapters.")
for c in STUDENT_CELLS.values():c.eval().requires_grad_(False)
with installed(GROUP_STARTS):
    selected_probe=compare_decisions(probe_items,probe_truth)
# Save the selected calibrated state before the blind audit, so GPU
# disconnections after this point do not lose the learned parameters.
probe_state=OUTPUT_DIR/"v39_probe_selected_adapters.pt"
torch.save(dict(
    source_model=SOURCE_MODEL,rank=CENN_RANK,shared=USE_SHARED_CELL,
    groups=GROUPS,
    probe=selected_probe,
    adapters={g:state_cpu(c) for g,c in STUDENT_CELLS.items()},
    selected_scales={g:list(c.enabled_scales) for g,c in STUDENT_CELLS.items()}
),probe_state)
print("Probe-selected combined hybrid:",selected_probe)
print("Saved pre-audit trained adapter state:",probe_state)


In [ ]:
#@title 11. Final blind AUDIT, CeNN necessity, and matched CUDA speed (full model AND group)
# Do not choose checkpoints or hyperparameters by these untouched AUDIT outcomes.
@torch.inference_mode()
def cuda_ms(fn,warmup=4,repeats=BENCH_REPEATS):
    with torch.autocast("cuda",dtype=amp_dtype):
        for _ in range(warmup):fn()
        torch.cuda.synchronize()
        times=[]
        for _ in range(repeats):
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            a.record();fn();z.record();z.synchronize()
            times.append(float(a.elapsed_time(z)))
    return dict(median_ms=float(np.median(times)),
                p95_ms=float(np.percentile(times,95)),repeats=len(times))

# Exact same batched token IDs and marker distributions in every comparison.
timing_batches=[batch(probe_items,0,BATCH),
                batch(probe_items,17,BATCH)]
timing_info=[(int(b["input_ids"].shape[0]),
              int(b["input_ids"].shape[1])) for b in timing_batches]
native_full=[cuda_ms(lambda b=b:teacher(**b)) for b in timing_batches]
with installed(GROUP_STARTS):
    compressed_full=[cuda_ms(lambda b=b:teacher(**b)) for b in timing_batches]
    blind_audit=compare_decisions(audit_items,audit_truth)
    for c in STUDENT_CELLS.values():c.spatial_enabled=False
    try:
        audit_no_spatial=compare_decisions(audit_items,audit_truth)
    finally:
        for c in STUDENT_CELLS.values():c.spatial_enabled=True

full_speedups=[a["median_ms"]/max(b["median_ms"],1e-8)
               for a,b in zip(native_full,compressed_full)]
# Interleaved A/B/B/A on identical token shapes is more reliable than
# timing all native passes first and all hybrid passes afterward.
from contextlib import nullcontext
def compare_paired_full(b,repeats=6):
    outcomes={"native":[],"hybrid":[]}
    # Perform warmup for both modes before collecting timing samples.
    for variant in ("native","hybrid"):
        cm=installed(GROUP_STARTS) if variant=="hybrid" else nullcontext()
        with cm,torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
            for _ in range(3):teacher(**b)
        torch.cuda.synchronize()
    for rep in range(repeats):
        order=("native","hybrid","hybrid","native") if rep%2==0 \
              else ("hybrid","native","native","hybrid")
        for variant in order:
            cm=installed(GROUP_STARTS) if variant=="hybrid" else nullcontext()
            with cm,torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
                start=torch.cuda.Event(enable_timing=True)
                stop=torch.cuda.Event(enable_timing=True)
                start.record()
                teacher(**b)
                stop.record()
                stop.synchronize()
                outcomes[variant].append(float(start.elapsed_time(stop)))
    a=float(np.median(outcomes["native"]))
    c=float(np.median(outcomes["hybrid"]))
    return dict(native_ms=a,hybrid_ms=c,speedup=a/max(c,1e-8),
                native_p95_ms=float(np.percentile(outcomes["native"],95)),
                hybrid_p95_ms=float(np.percentile(outcomes["hybrid"],95)),
                repeats=repeats,token_length=int(b["input_ids"].shape[1]),
                batch_size=int(b["input_ids"].shape[0]))

paired_full=[compare_paired_full(b,repeats=6) for b in timing_batches]
print("Interleaved matched whole-model CUDA performance:",paired_full)

print("AUDIT final hybrid:",blind_audit)
print("CeNN SPATIAL OFF audit:",audit_no_spatial)
print("Matched timings (B,T):",timing_info)
print("Original full model:",native_full)
print("9x global + two-step CeNN:",compressed_full)
print("Whole-model speedups:",full_speedups)

# Stage-level measurement: from post-global output to post-slide-2 output.
segment_timings=[]
for g,_,_ in GROUPS:
    snap=capture_teacher_triplet(g,probe_items,1,"BENCH")
    e=to_gpu(snap[0])
    # Match the post-global hidden state, T and B exactly.
    # Original local layers need precise 4D attention/sliding masks supplied
    # by the encoder. Capture their kwargs for this same sample.
    sample=batch(probe_items,0,BATCH)
    got={}
    def get_kwargs(mod,args,kwargs):
        got.update(kwargs)
    hook=ORIGINAL_ENCODER_LAYERS[g+1].register_forward_pre_hook(
        get_kwargs,with_kwargs=True)
    try:
        with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
            teacher(**sample)
    finally:
        hook.remove()
    assert "sliding_window_mask" in got
    def native_two():
        z=ORIGINAL_ENCODER_LAYERS[g+1](e["x"],**got)[0]
        return ORIGINAL_ENCODER_LAYERS[g+2](z,**got)[0]
    def new_two():
        return STUDENT_CELLS[str(g)](e["x"],e["valid"])
    base_time=cuda_ms(native_two)
    cell_time=cuda_ms(new_two)
    segment_timings.append(dict(start=g,original_two=base_time,
                                cenn_two=cell_time,
                                speedup=base_time["median_ms"]/
                                max(cell_time["median_ms"],1e-8)))
    print("G",g,"native 2 slides",base_time["median_ms"],
          "CeNN 2 steps",cell_time["median_ms"],
          "ratio",segment_timings[-1]["speedup"])
    del snap,e

# Acceptance requires repeated whole-model speed and downstream agreement,
# independent of the local output cosine.
accuracy_gate=(blind_audit["teacher_agreement"]>=.95 and
               blind_audit["gold_accuracy_drop"]<=.01)
speed_gate=all(row["speedup"]>1.0 for row in paired_full)
local_gate=all(group_training_summary[str(g)]["cosine"]>=.99
               for g,_,_ in GROUPS)
spatial_gain=blind_audit["teacher_agreement"]-audit_no_spatial["teacher_agreement"]
gates=dict(teacher_agreement_and_gold=accuracy_gate,
           end_to_end_faster_on_all_batches=speed_gate,
           isolated_group_cosine_ge_099=local_gate,
           full_accept=bool(accuracy_gate and speed_gate and local_gate))
print("CeNN spatial audit agreement gain:",spatial_gain)
print("Strict gates:",gates)


In [ ]:
#@title 12. Save lightweight adapter checkpoint, all 9 group reports, and plots
from datetime import datetime,timezone
import matplotlib.pyplot as plt
checkpoint=OUTPUT_DIR/"v39_global_cenn_two_sliding_3to1_EXPERIMENTAL.pt"
report_path=OUTPUT_DIR/"v39_global_cenn_two_sliding_report.json"
artifact=dict(
    version="3.9-global-plus-shared-cenn-two-sliding",
    source_checkpoint=SOURCE_MODEL,
    group_triplets=GROUPS,
    final_unpaired_full=FULL_LAYERS[-1],
    local_cell_rank=CENN_RANK,
    shared_cell_within_group=USE_SHARED_CELL,
    spatial_scales=[(7,1),(7,4),(9,16)],
    training_steps_per_group=GROUP_STEPS,
    end2end_calibration=RUN_END2END_CALIBRATION,
    source_global_blocks_unchanged=True,
    trained_sliding_blocks_per_group=2,
    actual_active_encoder_layers=10,
    original_encoder_slots=28,
    teacher_probe_post_distillation=probe_pre_calibration,
    selected_probe=selected_probe,
    blind_audit=blind_audit,
    blind_cenn_spatial_off=audit_no_spatial,
    cenn_spatial_ablation=cenn_ablation,
    probe_selected_spatial_scales=scale_pruning,
    adjacent_sliding_weight_similarity=group_weight_similarity,
    optional_pointwise_control=pointwise_control,
    group_training_probe=group_training_summary,
    task_calibration_history=calibration_history,
    segment_cuda=segment_timings,
    native_full_cuda=native_full,
    hybrid_full_cuda=compressed_full,
    hardware=torch.cuda.get_device_name(0),
    actual_full_model_speedups=full_speedups,
    paired_interleaved_full_model_cuda=paired_full,
    original_teacher_parameters=teacher_parameters,
    trained_adapter_parameters=student_parameters,
    bypassed_sliding_parameters=removed_slide_parameters,
    theoretical_exported_parameter_count=
        teacher_parameters-removed_slide_parameters+student_parameters,
    gates=gates,
    official_test_used=False,
    created_utc=datetime.now(timezone.utc).isoformat(),
    caveats=[
        "One exact pretrained full-attention block per group is preserved.",
        "CeNN's two steps approximate two complete sliding blocks including their MLPs.",
        "True full-model accuracy and speed cannot be predicted from local cosine.",
        "Native sliding weights need not be similar; measured parameter cosine is not functional equivalence.",
        "No original pretrained checkpoint has been modified.",
        "GPU timings depend on hardware, input length, batch size and execution backend.",
    ]
)
report_path.write_text(json.dumps(artifact,indent=2,default=str),
                       encoding="utf-8")
torch.save(dict(
    config=dict(hidden=int(teacher.encoder.config.hidden_size),
                rank=CENN_RANK,shared=USE_SHARED_CELL,
                groups=GROUPS,scales=[(7,1),(7,4),(9,16)]),
    source_model=SOURCE_MODEL,
    adapter_state={g:state_cpu(c) for g,c in STUDENT_CELLS.items()},
    selected_spatial_scales={g:list(c.enabled_scales)
                             for g,c in STUDENT_CELLS.items()},
    accepted_for_deployment=bool(gates["full_accept"]),
),checkpoint)
print("RESEARCH CHECKPOINT:",checkpoint)
print("REPORT:",report_path)
print("Training only CeNN adapters; all teacher weights unchanged.")
print("Acceptance:",gates)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot([g for g,_,_ in GROUPS],
             [group_training_summary[str(g)]["cosine"] for g,_,_ in GROUPS],
             "o-",label="PROBE segment cosine")
axes[0].axhline(.99,linestyle="--",label="Goal")
axes[0].set(xlabel="Triple starting index",ylabel="Segment output cosine",
            title="Distilled group fidelity")
axes[0].legend()
axes[1].bar(["native","Global+CeNN"],
            [float(np.median([t["median_ms"] for t in native_full])),
             float(np.median([t["median_ms"] for t in compressed_full]))])
axes[1].set(ylabel="Median full-model GPU ms",title="Matched model latency")
fig.tight_layout()
fig_path=OUTPUT_DIR/"v39_global_cenn_quality_speed.png"
fig.savefig(fig_path,dpi=145,bbox_inches="tight")
plt.show()
print("Figure:",fig_path)
